# Group contrast: one subset against the rest

A reader often holds one subset of the rows in mind — one region,
one cohort, one licence type — and asks how that subset differs
from the rest of the table. Profiling the subset alone cannot
answer the question, because it loses the comparison; the report
diff compares two versions of a table, not two populations inside
one table. The group contrast is the comparison stated directly.

Declare the group through `hints`: either a boolean column, whose
true rows are the group, or a column and a value, so the group is
the rows where the column equals the value. The remaining rows are
the comparison population. The `group_contrast` detector is
hint-gated: it runs only when a contrast is declared, at every
budget, and a run that declares none is unchanged.

Each other column is compared in the terms its kind supports.
Numeric columns are compared by their means and medians, with the
mean difference scaled by the pooled standard deviation (Cohen's
d). Categorical columns are compared by level shares, boolean
columns by the share of true values. A difference is reported
only when the sizes support it: at least 10 rows on each side and
at least 10 known values on each side for the column, an effect of
at least 0.5 pooled standard deviations for a numeric column or a
share difference of at least 0.10 for a categorical or boolean
column, and in every case a difference at least 3 times its
standard error. A small group must therefore differ by a lot
before anything is stated. Findings are ranked by effect size and
capped at 10: the output is a summary of how the two populations
differ, not an exhaustive table. The group-defining column itself
is never contrasted.

The example builds a usage table whose 40 `north` rows were given
an extra 200 tokens each and the pro plan at three times the rest
of the table's rate, on top of patterns both populations share.

In [ ]:
import numpy as np
import polars as pl
import onceover

rows = np.arange(2_040)
is_north = rows < 40
usage = pl.DataFrame({
    "region": np.where(is_north, "north",
                       np.where(rows % 3 == 0, "south", "east")),
    "week": (rows % 52) + 1,
    "tokens": 200 + (rows % 97) + np.where(is_north, 200, 0),
    "plan": np.where(is_north,
                     np.where(rows % 5 < 3, "pro", "free"),
                     np.where(rows % 5 == 0, "pro", "free")),
})

report = onceover.profile(
    usage, hints={"contrast": {"column": "region", "value": "north"}})
for finding in report.select(type="group_contrast"):
    print(finding["claim"])
print(report.coverage["group_contrast"]["detail"])

In [ ]:
import altair as alt

labelled = usage.with_columns(
    pl.when(pl.col("region") == "north")
    .then(pl.lit("north group"))
    .otherwise(pl.lit("rest of the table"))
    .alias("population"))
(alt.Chart(labelled).mark_bar(opacity=0.55)
 .encode(x=alt.X("tokens:Q", bin=alt.Bin(maxbins=30), title="Tokens"),
         y=alt.Y("count()", title="Rows"),
         color=alt.Color("population:N", title="Population"))
 .properties(title="Token counts: the north group sits one band higher",
             width=560, height=240))

## Reading the result

Each finding states the direction and the size in plain terms: the
group's mean against the rest's mean, with the difference also
given in pooled standard deviations, or the group's share of a
level against the rest's share, in percentage points. The evidence
carries the counts behind each statement — the rows and known
values on both sides, both means and medians for a numeric column,
and for a categorical column every level whose share moved by at
least 10 points, up to three. The `week` column, which follows the
same pattern in both populations, produces no finding: identical
distributions are silent, as are columns constant within both
populations.

The coverage entry is part of the result. It names the group, the
row counts on both sides, any rows set aside because the group
column is missing there, how many columns were evaluated, and how
many contrasts cleared the bars. When a group cannot be formed —
the column is not in the table, the value names no rows, the group
covers every assigned row, or a side holds fewer than 10 rows —
the pass does not run and the coverage entry states the reason
instead.

A contrast is descriptive. It says how two populations in this
table differ on the columns measured; it does not say why, and a
difference cleared by the bars is not evidence that group
membership causes it. The bars are fixed, so a group of 10 rows
needs a far larger difference than a group of 10,000 before
anything is stated, and differences below the floors are not
reported at all rather than reported as small. The contrast is
computed from the same extracted table the rest of the profile
uses, so on a sampled source it describes the sample the report
describes.